# 06 Model Diagnosis

对当前风险分类模型进行不平衡多分类诊断，输出核心指标、诊断图和 markdown 诊断报告。


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.models.model_evaluator import CreditModelEvaluator

ARTIFACT_DIR = PROJECT_ROOT / "src" / "models" / "artifacts"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["正常类", "关注类", "次级类", "可疑类", "损失类"]


In [ ]:
preprocessor = joblib.load(ARTIFACT_DIR / "preprocessor.joblib")
selector = joblib.load(ARTIFACT_DIR / "selector.joblib")
risk_classifier = joblib.load(ARTIFACT_DIR / "risk_classifier.joblib")

def resolve_split_path(split_name: str) -> Path:
    purified_path = DATA_DIR / f"purified_{split_name}.csv"
    return purified_path if purified_path.exists() else DATA_DIR / f"{split_name}.csv"

test_df = pd.read_csv(resolve_split_path("test"), low_memory=False)

X_test_processed = preprocessor.transform(test_df)
X_test_selected = selector.transform(X_test_processed)
y_test = test_df["preloan_risk_label"].astype(int)

evaluator = CreditModelEvaluator(
    model=risk_classifier,
    X_test=X_test_selected,
    y_test=y_test,
    class_names=CLASS_NAMES,
)
metrics = evaluator.evaluate_imbalanced_multiclass()
active_class_names = metrics["class_names"]
metrics["generated_figures"] = {
    "confusion_matrix": evaluator.plot_confusion_matrix(),
    "class_metrics": evaluator.plot_classification_metrics(),
    "multiclass_roc_curve": evaluator.plot_roc_curve_multiclass(),
}
metrics


In [ ]:
named_report = metrics["classification_report_named"]
per_class_rows = []
for class_name in active_class_names:
    class_metrics = named_report.get(class_name, {})
    per_class_rows.append(
        {
            "类别": class_name,
            "Precision": round(float(class_metrics.get("precision", 0.0)), 6),
            "Recall": round(float(class_metrics.get("recall", 0.0)), 6),
            "F1-Score": round(float(class_metrics.get("f1-score", 0.0)), 6),
            "Support": int(class_metrics.get("support", 0.0)),
            "AUC_OVR": round(float(metrics["auc_roc_per_class"].get(class_name, float("nan"))), 6),
            "KS": round(float(metrics["ks_per_class"].get(class_name, float("nan"))), 6),
            "不良捕捉率": round(float(metrics["bad_rate_capture_per_class"].get(class_name, float("nan"))), 6),
        }
    )

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)
summary_text = (
    f"**Macro-F1:** {metrics['macro_f1']:.6f}  \n"
    f"**Weighted-F1:** {metrics['weighted_f1']:.6f}  \n"
    f"**OVO Macro AUC:** {metrics['auc_roc_ovo_macro']:.6f}  \n"
    f"**KS Mean:** {metrics['ks_value']:.6f}"
)
display(Markdown(summary_text))


In [ ]:
def build_diagnosis_conclusion(metric_payload: dict, active_names: list[str]) -> tuple[list[str], list[str]]:
    report = metric_payload["classification_report_named"]
    conclusions = []
    triggers = []

    zero_recall_classes = [
        class_name
        for class_name in active_names
        if float(report.get(class_name, {}).get("recall", 0.0)) == 0.0
    ]
    if zero_recall_classes:
        triggers.append("严重退化：存在召回率为0的类别 -> " + ", ".join(zero_recall_classes))

    normal_prediction_ratio = float(metric_payload["predicted_class_distribution"].get("正常类", 0.0))
    non_normal_recalls = [
        float(report.get(class_name, {}).get("recall", 0.0))
        for class_name in active_names
        if class_name != "正常类"
    ]
    if normal_prediction_ratio > 0.9 and non_normal_recalls and all(value < 0.1 for value in non_normal_recalls):
        triggers.append("模型偷懒：正常类预测占比超过90%，且其他类别召回率均低于10%")

    f1_gap = abs(float(metric_payload["weighted_f1"]) - float(metric_payload["macro_f1"]))
    if f1_gap > 0.3:
        triggers.append(
            f"严重不平衡退化：Weighted-F1 与 Macro-F1 差值为 {f1_gap:.6f}，超过 0.3"
        )

    if triggers:
        conclusions.append("当前模型存在明显的不平衡退化或主类挤压现象。")
    else:
        conclusions.append("当前模型未触发预设的严重退化规则，但仍需结合混淆矩阵观察类别边界。")

    return conclusions, triggers

conclusions, triggers = build_diagnosis_conclusion(metrics, active_class_names)
display(Markdown("## 自动诊断结论"))
for item in conclusions:
    display(Markdown(f"- {item}"))
for item in triggers:
    display(Markdown(f"- {item}"))


In [ ]:
report_path = DOCS_DIR / "model_diagnosis_report.md"
confusion_rows = metrics["confusion_matrix"]
report_lines = [
    "# 模型诊断报告",
    "",
    "## 核心指标",
    f"- Macro-F1: {metrics['macro_f1']:.6f}",
    f"- Weighted-F1: {metrics['weighted_f1']:.6f}",
    f"- OVO Macro AUC: {metrics['auc_roc_ovo_macro']:.6f}",
    f"- OVO Weighted AUC: {metrics['auc_roc_ovo_weighted']:.6f}",
    f"- 平均 KS: {metrics['ks_value']:.6f}",
    "",
    "## 各类别指标",
    "",
    "| 类别 | Precision | Recall | F1-Score | Support | AUC_OVR | KS | 不良捕捉率 |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for row in per_class_rows:
    report_lines.append(
        f"| {row['类别']} | {row['Precision']:.6f} | {row['Recall']:.6f} | {row['F1-Score']:.6f} | {row['Support']} | {row['AUC_OVR']:.6f} | {row['KS']:.6f} | {row['不良捕捉率']:.6f} |"
    )

report_lines.extend(
    [
        "",
        "## 混淆矩阵",
        "",
        "| 真实\预测 | " + " | ".join(active_class_names) + " |",
        "|---|" + "---|" * len(active_class_names),
    ]
)
for class_name, row in zip(active_class_names, confusion_rows):
    values = " | ".join(str(int(value)) for value in row)
    report_lines.append(f"| {class_name} | {values} |")

report_lines.extend([
    "",
    "## 自动诊断结论",
])
for item in conclusions:
    report_lines.append(f"- {item}")
for item in triggers:
    report_lines.append(f"- {item}")

report_lines.extend(
    [
        "",
        "## 诊断图表",
        f"- 混淆矩阵: {metrics['generated_figures']['confusion_matrix']}",
        f"- 类别指标对比: {metrics['generated_figures']['class_metrics']}",
        f"- 多分类 ROC: {metrics['generated_figures']['multiclass_roc_curve']}",
    ]
)

report_path.write_text("\n".join(report_lines), encoding="utf-8")
display(Markdown(f"诊断报告已保存到 `{report_path.relative_to(PROJECT_ROOT)}`"))
